In [7]:

import seaborn as sns
from IPython.display import HTML, display

df = sns.load_dataset('iris')

print(df.head())
print(df.describe())


   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa
       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.057333      3.758000     1.199333
std        0.828066     0.435866      1.765298     0.762238
min        4.300000     2.000000      1.000000     0.100000
25%        5.100000     2.800000      1.600000     0.300000
50%        5.800000     3.000000      4.350000     1.300000
75%        6.400000     3.300000      5.100000     1.800000
max        7.900000     4.400000      6.900000     2.500000


In [ ]:
import matplotlib.pyplot as plt

features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, feature in zip(axes, features):
    sns.histplot(df[feature], kde=True, ax=ax)
    ax.set_title(feature)
    ax.set_xlabel('')

plt.tight_layout()

In [3]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

import torch 
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


# select features and encode labels
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values.astype('float32')
X[0:5]

y = df['species'].astype('category').cat.codes.values.astype('int64')  # 0,1,2

# train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# convert to torch tensors
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train).long()
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test).long()

# create datasets and dataloaders
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# expose loaders for training
train_loader, test_loader

train_loader

(<torch.utils.data.dataloader.DataLoader at 0x22fb002b4d0>,
 <torch.utils.data.dataloader.DataLoader at 0x22fafcebbf0>)

In [ ]:
from torch import nn
from sklearn.metrics import confusion_matrix
import numpy as np

# simple classifier
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# train
epochs = 100
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

# evaluate
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

test_accuracy = (all_preds == all_targets).mean()
cm = confusion_matrix(all_targets, all_preds)

print(f"Test accuracy: {test_accuracy:.4f}")
print("Confusion matrix:")
print(cm)

Test accuracy: 1.0000
Confusion matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
